In [1]:
import sqlite3

# Verbindung zur SQLite-Datenbank (Datei wird erstellt, falls sie nicht existiert)
conn = sqlite3.connect("internatsDB.db")
cursor = conn.cursor()

# Foreign Keys aktivieren (wichtig in SQLite!)
cursor.execute("PRAGMA foreign_keys = ON;")

# =========================
# Tabellen erstellen
# =========================

cursor.execute("""
CREATE TABLE IF NOT EXISTS Internat (
    ID INTEGER PRIMARY KEY AUTOINCREMENT,
    Name TEXT NOT NULL UNIQUE,
    Gruendungsjahr DATE
);
""")

cursor.execute("""
CREATE TABLE IF NOT EXISTS Lehrer (
    ID INTEGER PRIMARY KEY AUTOINCREMENT,
    Name TEXT NOT NULL,
    Spezialisierung TEXT
);
""")

cursor.execute("""
CREATE TABLE IF NOT EXISTS Schueler (
    ID INTEGER PRIMARY KEY AUTOINCREMENT,
    Name TEXT NOT NULL,
    Geburtsdatum DATE NOT NULL,
    Internat_ID INTEGER NOT NULL,
    FOREIGN KEY (Internat_ID)
        REFERENCES Internat(ID)
        ON DELETE CASCADE
        ON UPDATE CASCADE
);
""")

cursor.execute("""
CREATE TABLE IF NOT EXISTS Unterricht (
    ID INTEGER PRIMARY KEY AUTOINCREMENT,
    Fach TEXT NOT NULL,
    Schueler_ID INTEGER NOT NULL,
    Lehrer_ID INTEGER NOT NULL,
    FOREIGN KEY (Schueler_ID)
        REFERENCES Schueler(ID)
        ON DELETE CASCADE,
    FOREIGN KEY (Lehrer_ID)
        REFERENCES Lehrer(ID)
        ON DELETE CASCADE
);
""")

cursor.execute("""
CREATE TABLE IF NOT EXISTS Projekte (
    ID INTEGER PRIMARY KEY AUTOINCREMENT,
    Name TEXT NOT NULL,
    Beschreibung TEXT,
    Internat_ID INTEGER NOT NULL,
    Betreuer_ID INTEGER,
    FOREIGN KEY (Internat_ID)
        REFERENCES Internat(ID)
        ON DELETE CASCADE,
    FOREIGN KEY (Betreuer_ID)
        REFERENCES Lehrer(ID)
        ON DELETE SET NULL
);
""")

# =========================
# Beispiel-Daten einfügen
# =========================

cursor.executemany("""
INSERT OR IGNORE INTO Internat (Name, Gruendungsjahr)
VALUES (?, ?);
""", [
    ("Internat Schlossberg", "1995-09-01"),
    ("Internat Seeblick", "2001-08-15")
])

cursor.executemany("""
INSERT INTO Lehrer (Name, Spezialisierung)
VALUES (?, ?);
""", [
    ("Herr Mueller", "Mathematik"),
    ("Frau Schmidt", "Informatik"),
    ("Herr Weber", "Biologie")
])

cursor.executemany("""
INSERT INTO Schueler (Name, Geburtsdatum, Internat_ID)
VALUES (?, ?, ?);
""", [
    ("Max Mustermann", "2008-05-12", 1),
    ("Lisa Klein", "2009-11-03", 1),
    ("Tom Becker", "2007-02-20", 2)
])

cursor.executemany("""
INSERT INTO Unterricht (Fach, Schueler_ID, Lehrer_ID)
VALUES (?, ?, ?);
""", [
    ("Mathematik", 1, 1),
    ("Informatik", 1, 2),
    ("Biologie", 2, 3),
    ("Mathematik", 3, 1)
])

cursor.executemany("""
INSERT INTO Projekte (Name, Beschreibung, Internat_ID, Betreuer_ID)
VALUES (?, ?, ?, ?);
""", [
    ("Robotik AG", "Bau und Programmierung von Robotern", 1, 2),
    ("Umweltprojekt", "Analyse der Wasserqualitaet", 2, 3)
])

# Änderungen speichern
conn.commit()

# =========================
# Test-Abfrage
# =========================

print("\n--- Schueler mit Internat ---")
cursor.execute("""
SELECT Schueler.Name, Internat.Name
FROM Schueler
JOIN Internat ON Schueler.Internat_ID = Internat.ID;
""")

for row in cursor.fetchall():
    print(f"Schueler: {row[0]} | Internat: {row[1]}")

# Verbindung schließen
conn.close()

print("\nDatenbank erfolgreich erstellt und befüllt!")


--- Schueler mit Internat ---
Schueler: Max Mustermann | Internat: Internat Schlossberg
Schueler: Lisa Klein | Internat: Internat Schlossberg
Schueler: Tom Becker | Internat: Internat Seeblick

Datenbank erfolgreich erstellt und befüllt!
